In [2]:
"""05d_house_crossval.py — confirm the schema fix and cross-validate Artemis Group in House LDA."""
import duckdb

con = duckdb.connect("../output/investigation.duckdb", read_only=True)

print("=== Schema check ===")
schema = {r[0]: r[1] for r in con.execute("DESCRIBE house_lobbyists").fetchall()}
print(f"house_lobbyists.lobbyist_name type: {schema['lobbyist_name']}")
print()

print("=== Bridenstine in House LDA ===")
rows = con.execute("""
    SELECT hf.client_name, hf.filing_year, hf.filing_period,
           hf.house_id, hf.senate_id, hf.income
    FROM house_lobbyists hl
    JOIN house_filings hf USING (house_id)
    WHERE hl.lobbyist_name ILIKE '%bridenstine%'
    ORDER BY hf.filing_year, hf.filing_period
""").fetchall()
print(f"Found {len(rows)} House filings naming Bridenstine")
for row in rows[:20]:
    print(row)
print()

print("=== All Artemis Group filings in House LDA ===")
rows = con.execute("""
    SELECT hf.client_name, hf.filing_year, hf.filing_period,
           hf.house_id, hf.senate_id, hf.income
    FROM house_filings hf
    WHERE hf.org_name ILIKE '%artemis group%'
    ORDER BY hf.filing_year, hf.filing_period, hf.client_name
""").fetchall()
print(f"Found {len(rows)} House filings registered by Artemis Group")
for row in rows[:50]:
    print(row)
print()

print("=== Artemis Group lobbyists in House LDA, ranked by filing count ===")
rows = con.execute("""
    SELECT hl.lobbyist_name,
           COUNT(DISTINCT hf.house_id) AS filings,
           COUNT(DISTINCT hf.client_name) AS clients
    FROM house_lobbyists hl
    JOIN house_filings hf USING (house_id)
    WHERE hf.org_name ILIKE '%artemis group%'
      AND hl.lobbyist_name IS NOT NULL
    GROUP BY hl.lobbyist_name
    ORDER BY filings DESC
""").fetchall()
for row in rows:
    print(row)
print()

print("=== Senate↔House reconciliation check ===")
# How many Senate Artemis Group filings have a matching House filing via senate_id?
rows = con.execute("""
    WITH senate_artemis AS (
        SELECT DISTINCT filing_uuid, client_name, filing_year, filing_period
        FROM senate_filings
        WHERE registrant_name ILIKE '%artemis group%'
    ),
    house_artemis AS (
        SELECT DISTINCT senate_id, client_name, filing_year, filing_period
        FROM house_filings
        WHERE org_name ILIKE '%artemis group%'
          AND senate_id IS NOT NULL
    )
    SELECT
        (SELECT COUNT(*) FROM senate_artemis) AS senate_filings,
        (SELECT COUNT(*) FROM house_artemis) AS house_filings_with_senate_id,
        (SELECT COUNT(DISTINCT senate_id) FROM house_artemis) AS distinct_senate_ids_in_house
""").fetchall()
for row in rows:
    print(row)

con.close()

=== Schema check ===
house_lobbyists.lobbyist_name type: VARCHAR

=== Bridenstine in House LDA ===
Found 0 House filings naming Bridenstine

=== All Artemis Group filings in House LDA ===
Found 115 House filings registered by Artemis Group
('Agile Space Industries', 2024, 'Q4', '301647981', '401108974-61848', 60000.0)
('All Points LLC', 2024, 'Q4', '301647982', '401108974-61849', 50000.0)
('Choctaw Nation of Oklahoma', 2024, 'Q4', '301647985', '401108974-61850', 60000.0)
('Frontier Electronic Systems Corporation', 2024, 'Q4', '301647986', '401108974-61851', 50000.0)
('Frontier Electronic Systems Corporation', 2024, 'Q4', '301683847', '401108974-61851', 50000.0)
('Impulse Space', 2024, 'Q4', '301650334', '401108974-56706', 40000.0)
('Impulse Space', 2024, 'Q4', '301656046', '401108974-61852', 40000.0)
('Oklahoma Space Industry Development Authority (OSIDA)', 2024, 'Q4', '301647987', '401108974-61853', 50000.0)
('Redwire', 2024, 'Q4', '301647988', '401108974-61854', 30000.0)
('Special Ae

In [4]:
# notebooks/05e_house_lobbyist_diagnostic.py
import duckdb

con = duckdb.connect("../output/investigation.duckdb", read_only=True)

# Pick one Artemis Group House filing and inspect its raw data
print("=== Sample Artemis Group House filing ===")
row = con.execute("""
    SELECT house_id, client_name, source_path
    FROM house_filings
    WHERE org_name ILIKE '%artemis group%'
    LIMIT 1
""").fetchone()
print(row)
house_id = row[0]
print()

print(f"=== Lobbyist rows for house_id={house_id} in our DB ===")
for r in con.execute(f"""
    SELECT *
    FROM house_lobbyists
    WHERE house_id = '{house_id}'
""").fetchall():
    print(r)
print()

print("=== How many Artemis House filings have ANY lobbyist rows? ===")
print(con.execute("""
    SELECT
        COUNT(DISTINCT hf.house_id) AS total_artemis_filings,
        COUNT(DISTINCT hl.house_id) AS filings_with_lobbyists
    FROM house_filings hf
    LEFT JOIN house_lobbyists hl USING (house_id)
    WHERE hf.org_name ILIKE '%artemis group%'
""").fetchone())
print()

print("=== Pick the source XML of that filing and let's look at it directly ===")
print(f"Open this file: {row[2]}")

=== Sample Artemis Group House filing ===
('301647981', 'Agile Space Industries', 'data\\data\\house\\2024_4thQuarter_XML\\301647981.xml')

=== Lobbyist rows for house_id=301647981 in our DB ===
('301647981', None, 'Member, United States House of Representatives; Administrator, NASA', 'N', 'data\\data\\house\\2024_4thQuarter_XML\\301647981.xml')
('301647981', None, '\u200bDistrict Director, United States House of Representatives, Hon. Bridenstine; Chief of Staff, NASA', 'N', 'data\\data\\house\\2024_4thQuarter_XML\\301647981.xml')

=== How many Artemis House filings have ANY lobbyist rows? ===
(115, 105)

=== Pick the source XML of that filing and let's look at it directly ===
Open this file: data\data\house\2024_4thQuarter_XML\301647981.xml


In [6]:
import duckdb
con = duckdb.connect("../output/investigation.duckdb", read_only=True)

print("=== House activities: how many have ali_code populated? ===")
print(con.execute("""
    SELECT COUNT(*) AS total,
           COUNT(ali_code) AS with_code,
           COUNT(description) AS with_desc
    FROM house_activities
""").fetchone())

print()
print("=== Sample 5 random house_activities rows ===")
for row in con.execute("""
    SELECT house_id, activity_idx, ali_code, description
    FROM house_activities
    USING SAMPLE 5 ROWS
""").fetchall():
    print(row)

=== House activities: how many have ali_code populated? ===
(781004, 45535, 735469)

=== Sample 5 random house_activities rows ===
('301420261', 0, None, 'Local government related issues and initiatives, including FY 2023 appropriations.')
('301495260', 2, None, 'SRFs, Farm Bill, STAG, Infrastructure Funding, PFAS, CWA, SDWA, Appropriations, NRCS, EQIP, RCPP, Agriculture, Reclamation, Infrastructure Funding.')
('301637191', 2, None, 'S. 615/HR 1293 Cabin Air Quality Act would protect airline pilots, flight attendants, and passengers from toxic cabin air. \n\nHR 3576/S. 1772 - Air PUMP Act would address the need for clear rules and regulations\nallowing flight crews to safely pump aboard aircraft\n\nIssues related to Sustainable Aviation Fuel Tax Credits\n\nIssues related to Child Restraint Seats would provide a seat for every passenger including small children/infants\n\nIssue related to access to naloxone on Aircraft\n\nEnding the Part 135/380 exemption for scheduled charter (JSX)\n\n

In [2]:
import duckdb
con = duckdb.connect("../output/investigation.duckdb", read_only=True)

print("Bridenstine in House (should be > 0 now):")
print(con.execute("""
    SELECT COUNT(*) FROM house_lobbyists WHERE lobbyist_name ILIKE '%bridenstine%'
""").fetchone())

print("Top 10 House lobbyists by filing count:")
for row in con.execute("""
    SELECT lobbyist_name, COUNT(DISTINCT house_id) AS filings
    FROM house_lobbyists
    WHERE lobbyist_name IS NOT NULL
    GROUP BY lobbyist_name
    ORDER BY filings DESC
    LIMIT 10
""").fetchall():
    print(row)

Bridenstine in House (should be > 0 now):
(125,)
Top 10 House lobbyists by filing count:
('Brian Ballard', 3021)
('Nichole Distefano', 2008)
('Bruce Mehlman', 2006)
('David Thomas', 1978)
('Sage Eastman', 1978)
('Alexander Perkins', 1969)
('Paul Thornell', 1968)
('Rosemary Gutierrez', 1961)
('Helen Tolar', 1930)
('Stephen Cote', 1924)


In [3]:
import duckdb
con = duckdb.connect("../output/investigation.duckdb", read_only=True)

print("=== House activities — ali_code coverage ===")
print(con.execute("""
    SELECT COUNT(*) AS total,
           COUNT(ali_code) AS with_code,
           ROUND(100.0 * COUNT(ali_code) / COUNT(*), 1) AS pct_coverage,
           COUNT(description) AS with_desc
    FROM house_activities
""").fetchone())

print()
print("=== House row counts (all 3 tables) ===")
for t in ("house_filings", "house_activities", "house_lobbyists"):
    print(f"  {t}: {con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]:,}")

=== House activities — ali_code coverage ===
(781004, 781004, 100.0, 735469)

=== House row counts (all 3 tables) ===
  house_filings: 409,640
  house_activities: 781,004
  house_lobbyists: 2,017,296
